### Import Needed Package

In [2]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings("ignore")

### Load Dataset

In [4]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [5]:
data = pd.read_csv("/content/drive/MyDrive/Lab. AI/IMDB Dataset.csv")

In [6]:
data.head()

,review,sentiment
0,One of the other reviewers has mentioned that ...,positive
1,A wonderful little production. <br /><br />The...,positive
2,I thought this was a wonderful way to spend ti...,positive
3,Basically there's a family where a little boy ...,negative
4,"Petter Mattei's ""Love in the Time of Money"" is...",positive


In [7]:
data.shape

(50000, 2)

In [8]:
type(data)

pandas.core.frame.DataFrame

In [9]:
data.tail()

,review,sentiment
49995,I thought this movie did a down right good job...,positive
49996,"Bad plot, bad dialogue, bad acting, idiotic di...",negative
49997,I am a Catholic taught in parochial elementary...,negative
49998,I'm going to have to disagree with the previou...,negative
49999,No one expects the Star Trek movies to be high...,negative


In [10]:
data["sentiment"].value_counts()

,count
sentiment,
positive,25000
negative,25000


### One Hot Encoding

#### Lavel Encoder

In [13]:
data.replace({"sentiment": {"positive": 1, "negative": 0}}, inplace=True)

In [14]:
data.head()

,review,sentiment
0,One of the other reviewers has mentioned that ...,1
1,A wonderful little production. <br /><br />The...,1
2,I thought this was a wonderful way to spend ti...,1
3,Basically there's a family where a little boy ...,0
4,"Petter Mattei's ""Love in the Time of Money"" is...",1


### Data Preprocessing

In [16]:
from sklearn.model_selection import train_test_split
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Embedding, LSTM
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences

In [17]:
train_data, test_data = train_test_split(data, test_size = 0.2, random_state=42)

In [18]:
train_data.shape

(40000, 2)

In [19]:
test_data.shape

(10000, 2)

In [20]:
tokenizer = Tokenizer(num_words=5000)
tokenizer.fit_on_texts(train_data["review"])

In [21]:
X_train = pad_sequences(tokenizer.texts_to_sequences(train_data["review"]), maxlen=200)
X_test = pad_sequences(tokenizer.texts_to_sequences(test_data["review"]), maxlen=200)

In [22]:
X_train

array([[1935,    1, 1200, ...,  205,  351, 3856],
       [   3, 1651,  595, ...,   89,  103,    9],
       [   0,    0,    0, ...,    2,  710,   62],
       ...,
       [   0,    0,    0, ..., 1641,    2,  603],
       [   0,    0,    0, ...,  245,  103,  125],
       [   0,    0,    0, ...,   70,   73, 2062]], dtype=int32)

In [23]:
X_test

array([[   0,    0,    0, ...,  995,  719,  155],
       [  12,  162,   59, ...,  380,    7,    7],
       [   0,    0,    0, ...,   50, 1088,   96],
       ...,
       [   0,    0,    0, ...,  125,  200, 3241],
       [   0,    0,    0, ..., 1066,    1, 2305],
       [   0,    0,    0, ...,    1,  332,   27]], dtype=int32)

In [24]:
Y_train = train_data["sentiment"]
Y_test = test_data["sentiment"]

In [25]:
Y_train

,sentiment
39087,0
30893,0
45278,1
16398,0
13653,0
...,...
11284,1
44732,1
38158,0
860,1


In [26]:
Y_test

,sentiment
33553,1
9427,1
199,0
12447,1
39489,0
...,...
28567,0
25079,1
18707,1
15200,0


In [27]:
model = Sequential()
model.add(Embedding(input_dim = 5000, output_dim=128, input_length=200))
model.add(LSTM(128, dropout=0.2, recurrent_dropout=0.2))
model.add(Dense(1, activation="sigmoid"))

In [28]:
model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding (Embedding)           │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm (LSTM)                     │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

In [30]:
model.compile(loss="binary_crossentropy", optimizer="adam", metrics=["accuracy"])

In [31]:
model.fit(X_train, Y_train, batch_size=64, epochs=5, validation_split=0.2)

Epoch 1/5
500/500 ━━━━━━━━━━━━━━━━━━━━ 213s 410ms/step - accuracy: 0.7333 - loss: 0.5238 - val_accuracy: 0.8543 - val_loss: 0.3462
Epoch 2/5
500/500 ━━━━━━━━━━━━━━━━━━━━ 200s 401ms/step - accuracy: 0.8515 - loss: 0.3528 - val_accuracy: 0.8609 - val_loss: 0.3395
Epoch 3/5
500/500 ━━━━━━━━━━━━━━━━━━━━ 201s 403ms/step - accuracy: 0.8787 - loss: 0.3040 - val_accuracy: 0.8565 - val_loss: 0.3365
Epoch 4/5
500/500 ━━━━━━━━━━━━━━━━━━━━ 203s 405ms/step - accuracy: 0.8901 - loss: 0.2740 - val_accuracy: 0.8675 - val_loss: 0.3236
Epoch 5/5
500/500 ━━━━━━━━━━━━━━━━━━━━ 210s 420ms/step - accuracy: 0.9118 - loss: 0.2314 - val_accuracy: 0.8752 - val_loss: 0.3110


In [ ]:
model.fit(X_train, Y_train, batch_size=64, epochs=5, validation_split=0.2)

In [33]:
loss, accuracy = model.evaluate(X_test, Y_test)

313/313 ━━━━━━━━━━━━━━━━━━━━ 38s 119ms/step - accuracy: 0.8865 - loss: 0.2970


In [34]:
print(loss)

0.295708566904068


In [35]:
print(accuracy)

0.8855999708175659


### Building Predictive System

In [36]:
def predictive_system(review):
  sequence = tokenizer.texts_to_sequences([review])
  padded_sequence = pad_sequences(sequence, maxlen=200)
  prediction = model.predict(padded_sequence)
  sentiment = "positive" if prediction[0][0] > 0.5 else "negative"
  print(f"Review: {review}\nSentiment: {sentiment}\nPrediction: {prediction[0][0]}")

In [37]:
predictive_system("I loved this movie")

1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 648ms/step
Review: I loved this movie
Sentiment: positive
Prediction: 0.9608848094940186


In [38]:
predictive_system("I hated this movie")

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 155ms/step
Review: I hated this movie
Sentiment: negative
Prediction: 0.09832678735256195


In [44]:
predictive_system("I'm amazed by this movie")

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 196ms/step
Review: I'm amazed by this movie
Sentiment: positive
Prediction: 0.6387153267860413


In [ ]:
predictive_system("this ")

### Saving Model

In [45]:
model.save("model.h5")

In [46]:
import joblib
joblib.dump(tokenizer, "tokenizer.pkl")

['tokenizer.pkl']